# House Price Prediction — Ames, Iowa

Kaggle competition with 79 features describing residential homes. Goal is to predict SalePrice.

I started with a quick script (`housing_price_v1.py`) that just used `get_dummies` + a basic XGBoost — got ~$16k MAE. This notebook is the cleaned-up version with a proper sklearn Pipeline, better encoding, and a log-transformed target. Ended up around $13k MAE on validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
from xgboost import XGBRegressor

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

print(f'Train: {df.shape}')
print(f'Test:  {test_data.shape}')
df.head()

## EDA

### Target distribution

SalePrice is right-skewed (makes sense — more cheap houses than mansions). Log-transforming it helps the model since it won't be pulled around by the high-end outliers. Also makes the residuals more interpretable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title(f"Raw  (skew={df['SalePrice'].skew():.2f})")
axes[0].set_xlabel('SalePrice')

axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title(f"log1p  (skew={np.log1p(df['SalePrice']).skew():.2f})")
axes[1].set_xlabel('log(SalePrice)')

plt.tight_layout()
plt.show()

### Missing values

A lot of columns have high missingness but most of it isn't actually missing — `PoolQC = NaN` means no pool, `Alley = NaN` means no alley access, etc. The data documentation makes this clear. So imputing with most_frequent is fine for the categoricals rather than doing anything fancy.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 6))
missing_pct.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Missing (%)')
ax.set_title('Features with missing values')
plt.tight_layout()
plt.show()

# top offenders
pd.DataFrame({'count': missing, 'pct': missing_pct}).head(10)

### What actually correlates with price?

`OverallQual` is basically the whole ballgame at r=0.79. `GrLivArea` and `GarageCars` are next. `YearBuilt` is in there too which makes sense — newer houses fetch more.

In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop('Id')
corr = df[num_cols].corr()['SalePrice'].drop('SalePrice').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
corr.plot(kind='barh', ax=ax, color=corr.map(lambda x: 'steelblue' if x > 0 else 'coral'))
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with SalePrice')
plt.tight_layout()
plt.show()

In [ ]:
# top 3 features — quick visual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.boxplot(column='SalePrice', by='OverallQual', ax=axes[0])
plt.sca(axes[0])
plt.title('OverallQual vs SalePrice')
axes[0].set_xlabel('OverallQual')

axes[1].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.3, s=10, color='steelblue')
axes[1].set_title('GrLivArea vs SalePrice')
axes[1].set_xlabel('Above ground sq ft')

df.boxplot(column='SalePrice', by='GarageCars', ax=axes[2])
plt.sca(axes[2])
plt.title('GarageCars vs SalePrice')

plt.suptitle('')
plt.tight_layout()
plt.show()

### Neighbourhoods

Big spread across neighbourhoods. NridgHt and NoRidge are clearly the premium areas.

In [ ]:
nbhd_median = df.groupby('Neighborhood')['SalePrice'].median().sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
nbhd_median.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Median SalePrice')
ax.set_title('Median price by neighbourhood')
plt.tight_layout()
plt.show()

### Outliers

Two houses with GrLivArea > 4000 but prices way below what you'd expect. Checked — they're partial sales (SaleCondition = Partial), not normal market transactions. Removing them from training since they'd skew the regression. They don't appear in the test set anyway.

In [ ]:
outlier_mask = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)
outliers = df[outlier_mask]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.3, s=10, color='steelblue', label='Normal')
axes[0].scatter(outliers['GrLivArea'], outliers['SalePrice'], color='red', s=50, label='Flagged', zorder=5)
axes[0].legend()
axes[0].set_title('Before')
axes[0].set_xlabel('GrLivArea')

df_clean = df[~outlier_mask].copy()

axes[1].scatter(df_clean['GrLivArea'], df_clean['SalePrice'], alpha=0.3, s=10, color='steelblue')
axes[1].set_title('After removing outliers')
axes[1].set_xlabel('GrLivArea')

plt.tight_layout()
plt.show()

print(f'Removed {len(outliers)} rows')
print(outliers[['GrLivArea', 'SalePrice', 'SaleCondition', 'Neighborhood']])

## Feature Engineering

Started with the same features from v1 (TotalSF, HouseAge, TotalBath). Added a quality×area interaction in v2 since a 2000sqft house rated 9/10 is very different from a 2000sqft house rated 4/10 — the model doesn't naturally capture that without the explicit interaction.

In [ ]:
def add_features(df):
    df = df.copy()
    df['TotalSF']   = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge']  = df['YrSold'] - df['YearBuilt']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath']
    df['QualXSF']   = df['OverallQual'] * df['GrLivArea']
    df['GarageRatio'] = df['GarageArea'] / df['LotArea'].replace(0, np.nan)
    return df

X = add_features(df_clean.drop(['SalePrice'], axis=1))
y = np.log1p(df_clean['SalePrice'])

# check correlations for engineered features
eng = ['TotalSF', 'HouseAge', 'TotalBath', 'QualXSF', 'GarageRatio']
for f in sorted(eng, key=lambda f: -abs(X[f].corr(y))):
    print(f'  {f:<15} {X[f].corr(y):+.3f}')

QualXSF (r=0.84) and TotalSF (r=0.83) are both strong. GarageRatio adds almost nothing at 0.13 but it doesn't hurt so I'm leaving it in.

I also tried `QualXAge = OverallQual * HouseAge` thinking quality interacts with age (a well-built old house vs a cheap new one) but it only correlated at -0.41 and wasn't adding anything the model didn't already have from QualXSF. Dropped it.

In [ ]:
# tried this — didn't add much over QualXSF
qual_x_age = X['OverallQual'] * X['HouseAge']
print(f'QualXAge correlation: {qual_x_age.corr(y):+.3f}')  # -0.41, skipping it

## v1 vs v2 — why the pipeline matters

In v1 I did `pd.get_dummies` + `pd.align` which works but has a few problems:
- Ordinal columns like `KitchenQual` (Po < Fa < TA < Gd < Ex) get treated as unordered categories
- The imputer is fit before the split, so technically there's a small leakage on LotFrontage etc.
- Reapplying to test data is clunky

Quick sanity check — what did v1 actually get?

In [ ]:
# replicating v1 approach to compare
X_v1 = add_features(df_clean.drop(['SalePrice'], axis=1))
y_v1 = df_clean['SalePrice']  # no log transform in v1

X_tr_v1, X_val_v1, y_tr_v1, y_val_v1 = train_test_split(X_v1, y_v1, random_state=0)

# v1-style: mark high-missingness categoricals as 0/1, get_dummies the rest
for_imputation = ['LotFrontage', 'MasVnrArea', 'GarageYrBlt']
missing_cats = X_tr_v1.columns[X_tr_v1.isnull().any()].tolist()
for_encoding = [c for c in missing_cats if c not in for_imputation and X_tr_v1[c].dtype == 'object']

for col in for_encoding:
    X_tr_v1[col]  = X_tr_v1[col].apply(lambda x: 0 if pd.isna(x) else 1)
    X_val_v1[col] = X_val_v1[col].apply(lambda x: 0 if pd.isna(x) else 1)

X_tr_v1  = pd.get_dummies(X_tr_v1)
X_val_v1 = pd.get_dummies(X_val_v1)
X_tr_v1, X_val_v1 = X_tr_v1.align(X_val_v1, join='left', axis=1, fill_value=0)

imputer_v1 = SimpleImputer(strategy='median')
X_tr_v1  = pd.DataFrame(imputer_v1.fit_transform(X_tr_v1),  columns=X_tr_v1.columns)
X_val_v1 = pd.DataFrame(imputer_v1.transform(X_val_v1), columns=X_val_v1.columns)

model_v1 = XGBRegressor(n_estimators=500, random_state=42)
model_v1.fit(X_tr_v1, y_tr_v1)
v1_mae = mean_absolute_error(y_val_v1, model_v1.predict(X_val_v1))
print(f'v1 MAE: ${v1_mae:,.0f}')

## Preprocessing Pipeline (v2)

The big change: proper `OrdinalEncoder` for quality columns with the actual ordering from the data docs, and a full sklearn Pipeline so nothing leaks between train and validation.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

# ordinal categories from the data documentation
quality_order    = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_fin_order   = ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_exp_order   = ['No', 'Mn', 'Av', 'Gd']
garage_fin_order = ['Unf', 'RFn', 'Fin']
land_slope_order = ['Gtl', 'Mod', 'Sev']
functional_order = ['Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
paved_order      = ['N', 'P', 'Y']

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
    'HeatingQC', 'KitchenQual', 'FireplaceQu',
    'GarageQual', 'GarageCond', 'PoolQC',
    'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure',
    'GarageFinish', 'LandSlope', 'Functional', 'PavedDrive'
]
ordinal_categories = (
    [quality_order] * 10 +
    [bsmt_fin_order] * 2 +
    [bsmt_exp_order] +
    [garage_fin_order] +
    [land_slope_order] +
    [functional_order] +
    [paved_order]
)

numerical_cols   = X_train.select_dtypes(exclude='object').columns
categorical_cols = [
    c for c in X_train.select_dtypes(include='object').columns
    if c not in ordinal_cols
]

print(f'Numerical: {len(numerical_cols)}, Ordinal: {len(ordinal_cols)}, Categorical: {len(categorical_cols)}')

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), numerical_cols),
    ('ord', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(
            categories=ordinal_categories,
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ]), ordinal_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

## Baseline

Just predicting the mean — gives a floor to beat. If the model can't beat a mean predictor something is badly wrong.

In [ ]:
# custom scorer to get dollar MAE (not log-space MAE)
def dollar_mae(y_true, y_pred):
    return mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))

dollar_scorer = make_scorer(dollar_mae, greater_is_better=False)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))
])

baseline_scores = cross_val_score(baseline_pipeline, X, y, cv=cv, scoring=dollar_scorer)
baseline_mae = -baseline_scores.mean()
print(f'Baseline (mean predictor) MAE: ${baseline_mae:,.0f}')

## XGBoost

Hyperparameters: low learning rate (0.01) with lots of trees (2500) generally works better than high learning rate with fewer trees on tabular data. max_depth=4 keeps trees shallow enough to not overfit — tried 3 and 5, 4 was the sweet spot. subsample and colsample_bytree at 0.8 add some stochasticity which helps generalisation.

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=2500,
        learning_rate=0.01,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print('Done')

## Evaluation

In [ ]:
val_preds   = np.expm1(pipeline.predict(X_valid))
val_actuals = np.expm1(y_valid)
val_mae     = mean_absolute_error(val_actuals, val_preds)

cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring=dollar_scorer)
cv_mae    = -cv_scores.mean()
cv_std    = cv_scores.std()

print(f'Baseline CV MAE: ${baseline_mae:>10,.0f}')
print(f'v1 Val MAE:      ${v1_mae:>10,.0f}   (get_dummies, no log transform)')
print(f'v2 Val MAE:      ${val_mae:>10,.0f}')
print(f'v2 CV MAE:       ${cv_mae:>10,.0f}  ± ${cv_std:,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(val_actuals, val_preds, alpha=0.3, s=10, color='steelblue')
lims = [min(val_actuals.min(), val_preds.min()), max(val_actuals.max(), val_preds.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1, label='Perfect')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

residuals = val_actuals - val_preds
axes[1].scatter(val_preds, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals — no obvious pattern is good')

plt.tight_layout()
plt.show()

# anything the model is consistently wrong on?
worst = pd.DataFrame({'actual': val_actuals.values, 'predicted': val_preds, 'error': residuals.values})
worst = worst.reindex(worst['error'].abs().sort_values(ascending=False).index)
print('Largest errors:')
print(worst.head(5).to_string(index=False))

The residuals look pretty random which is what we want. The model tends to underestimate on the very high end — houses above $400k are trickier, probably because there are fewer of them in training and they have more idiosyncratic features.

In [ ]:
# feature importance
xgb_model = pipeline.named_steps['model']
ohe_names = (pipeline.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .named_steps['onehot']
                      .get_feature_names_out(categorical_cols))

feat_names  = list(numerical_cols) + ordinal_cols + list(ohe_names)
importances = pd.Series(xgb_model.feature_importances_, index=feat_names)
top20       = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
top20.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 feature importances')
plt.tight_layout()
plt.show()

# QualXSF should be near the top
print('Engineered feature ranks:')
for f in ['QualXSF', 'TotalSF', 'HouseAge', 'TotalBath', 'GarageRatio']:
    rank = importances.rank(ascending=False)[f]
    print(f'  {f:<15} rank {int(rank)}')

## Save and submit

In [ ]:
joblib.dump(pipeline, 'housing_price_pipeline.pkl')

# sanity check — reload and verify predictions match
loaded = joblib.load('housing_price_pipeline.pkl')
assert np.allclose(val_preds, np.expm1(loaded.predict(X_valid)))
print('Pipeline saved and verified')

In [ ]:
X_test     = add_features(test_data)
test_preds = np.expm1(pipeline.predict(X_test))

output = pd.DataFrame({'Id': test_data['Id'], 'SalePrice': test_preds})
output.to_csv('submission_v2.csv', index=False)

print(f'Saved {len(output)} predictions')
print(f"Price range: ${output['SalePrice'].min():,.0f} – ${output['SalePrice'].max():,.0f}")
output['SalePrice'].describe().apply(lambda x: f'${x:,.0f}')